# Run your own translation server

This runs the translator's server on your Modal account (free monthly
credit — Modal unlocks the full amount once you add a card).
Click Runtime → Run all, then paste the two values printed at the end into
the extension.

Before you start:

1. Create a free account at [modal.com](https://modal.com).
2. In Modal: Settings → API tokens → create a token.
3. On this page: click 🔑 Secrets (left sidebar) and add `MODAL_TOKEN_ID`
   and `MODAL_TOKEN_SECRET` with the two values.


## 1. Install

Installs the Modal tool and the packages the deploy reads. Takes a minute or two.


In [ ]:
%pip install -q modal==1.5.5 fastapi "uvicorn[standard]" onnxruntime numpy pillow opencv-python-headless
import shutil
assert shutil.which("modal"), "modal CLI not found after install — re-run this cell"
print("✓ ready")


## 2. Log in

Uses the two 🔑 Secrets from the left sidebar. Values are checked
before anything runs.


In [ ]:
import os, subprocess
try:
    from google.colab import userdata  # type: ignore  # Colab only
    def _get(k):
        try:
            return userdata.get(k)
        except Exception:
            return ""
except ImportError:  # local Jupyter fallback — typed, never saved
    import getpass
    def _get(k):
        return getpass.getpass(k + ": ")
tid = (_get("MODAL_TOKEN_ID") or "").strip()
tsec = (_get("MODAL_TOKEN_SECRET") or "").strip()
if not tid or not tsec:
    raise SystemExit("✗ add both values to the 🔑 Secrets sidebar (exact names "
                     "MODAL_TOKEN_ID / MODAL_TOKEN_SECRET), then re-run this cell")
if not tid.startswith("ak-"):
    raise SystemExit('✗ MODAL_TOKEN_ID should start with "ak-" — check the values')
if not tsec.startswith("as-"):
    raise SystemExit('✗ MODAL_TOKEN_SECRET should start with "as-" — check the values')
os.environ["MODAL_TOKEN_ID"] = tid
os.environ["MODAL_TOKEN_SECRET"] = tsec
r = subprocess.run(["modal", "token", "info"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("✗ Modal rejected the token — it may be revoked; create a new one "
                     "(Modal dashboard → Settings → API tokens), then re-run this cell.\n"
                     + r.stderr[-500:])
print("✓ Authenticated:")
print(r.stdout.strip()[-500:] or '(ok)')


## 3. Save the server files

Copies the server code here. Just run both cells.


In [ ]:
%%writefile modal_app.py
# Modal deployment of server/app.py: T4 GPU, scale-to-zero, Bearer auth.
# Deploy from inside server/: modal deploy modal_app.py
# (needs MODAL_TOKEN_ID/SECRET). app.py bakes into the image — each deploy
# is immutable, no runtime mounts.
import os
import sys

import modal

sys.path.insert(0, os.environ.get("PKG_DIR", os.path.dirname(os.path.abspath(__file__))))
# NOTE: top-level sibling import — resolved locally at deploy parse time
# (needs the full deps installed where you deploy from), baked copy used remotely.
from app import app as fastapi_app

CTD_URL = ("https://huggingface.co/lemondouble/lemon-manga-translator"
           "/resolve/main/onnx/comic-text-detector/ctd.onnx?download=true")
BABERU = "https://huggingface.co/genshiai-daichi/baberu-ocr/resolve/main"
BABERU_FILES = [
    ("onnx/vision_int4.onnx", "vision-int4.onnx"),
    ("onnx/decoder_prefill_int8.onnx", "baberu-prefill.onnx"),
    ("onnx/decoder_step_int8.onnx", "baberu-step.onnx"),
    ("tokenizer/vocab.json", "vocab.json"),
]

dl = [f"mkdir -p /models",
      f'curl -fL -o /models/ctd.onnx "{CTD_URL}"']
dl += [f'curl -fL -o /models/{dst} "{BABERU}/{src}?download=true"'
       for src, dst in BABERU_FILES]

image = (
    # nvidia runtime base: onnxruntime-gpu needs CUDA 13 + cuDNN 9 system
    # libs (libcublasLt.so.13) that debian_slim lacks — plain pip is not enough
    modal.Image.from_registry("nvidia/cuda:13.0.2-cudnn-runtime-ubuntu24.04",
                              add_python="3.12")
    .apt_install("curl")
    .pip_install("fastapi", "uvicorn[standard]", "onnxruntime-gpu",
                 "numpy", "pillow", "opencv-python-headless")
    .env({"ORT_PROVIDERS": "CUDAExecutionProvider,CPUExecutionProvider",
          "ORT_DEVICE": "cuda", "PKG_DIR": "/pkg"})
    .run_commands(*dl)
    # single file, not add_local_dir("."): deploy sources like Colab's /content
    # hold mutating internal files (.config/gce) that abort the build mid-snapshot
    .add_local_file("app.py", remote_path="/pkg/app.py")
)

app = modal.App("arn-manga")


@app.function(image=image, gpu=["T4", "L4"], timeout=600,
              secrets=[modal.Secret.from_name("arn-manga-key")])
@modal.asgi_app()
def api():
    import os

    from starlette.middleware.base import BaseHTTPMiddleware
    from starlette.responses import JSONResponse

    key = os.environ["ARN_API_KEY"]

    class Auth(BaseHTTPMiddleware):
        async def dispatch(self, request, call_next):
            if request.url.path in ("/", "/health"):
                return await call_next(request)
            if request.headers.get("authorization") != f"Bearer {key}":
                return JSONResponse({"ok": False, "error": "unauthorized"}, 401)
            return await call_next(request)

    fastapi_app.add_middleware(Auth)
    return fastapi_app


In [ ]:
%%writefile app.py
# Cloud inference for the manga translator: CTD text detection + Baberu OCR
# over HTTP, CPU-only (fits the HuggingFace free tier).
#
# Recipes ported 1:1 from src/iframe/worker.ts — same thresholds, same gates,
# same decode loop. Resize uses bilinear like canvas drawImage; exact pixels
# may differ from the browser path, so parity is VERIFIED (not assumed) by
# comparing boxes/texts against the local pipeline on real pages.
#
# Panels: v1 returns [] — the client falls back to banding ordering, the same
# path it takes when the panel model file is missing. No fidelity risk.
import asyncio
import io
import json
import os
import re
import time

import cv2
import numpy as np
import onnxruntime as ort
from fastapi import FastAPI, Query, Request
from fastapi.responses import JSONResponse
from PIL import Image

MODEL_DIR = os.environ.get("MODEL_DIR", "/models")
# EP chain: local default CPU; Modal sets "CUDAExecutionProvider,CPUExecutionProvider"
ORT_PROVIDERS = os.environ.get("ORT_PROVIDERS", "CPUExecutionProvider").split(",")

# ---- tunables: mirror src/iframe/worker.ts exactly ----
CTD_INPUT = 1024
CONF_THR = 0.35
NMS_THR = 0.35
MASK_THR = 0.3
MIN_SIZE = 12
LETTERBOX = (113, 113, 113)  # #717171
STRIP_ASPECT = 3
TILE_SIZE = 1200
TILE_OVERLAP = 180
COMP_GAP = 28

BABERU_MEAN = (0.485, 0.456, 0.406)
BABERU_STD = (0.229, 0.224, 0.225)
PAST_IN = [f"past_k{i}" for i in range(6)] + [f"past_v{i}" for i in range(6)]
PRESENT_OUT = [f"present_k{i}" for i in range(6)] + [f"present_v{i}" for i in range(6)]

app = FastAPI(title="arn-manga")
lock = asyncio.Lock()  # one inference at a time (2 vCPU, no oversubscription)
ctd = None
baberu = None  # {vis, pre, stp, bos, eos, id2ch, contentIds}
EPS = {}  # session -> provider chain (proves GPU placement in prod logs)


def _load(path):
    if not os.path.isfile(path):
        raise RuntimeError(f"model file missing: {path}")
    return path


OPTS = ort.SessionOptions()


def _sess(path, providers):
    t = time.perf_counter()
    s = ort.InferenceSession(_load(path), OPTS, providers=providers)
    print(f"session {os.path.basename(path)}: {(time.perf_counter()-t)*1000:.0f}ms",
          flush=True)
    return s


def load_models():
    global ctd, baberu
    print("onnxruntime:", ort.__version__,
          "available:", ort.get_available_providers(), flush=True)
    t_all = time.perf_counter()
    ctd = _sess(f"{MODEL_DIR}/ctd.onnx", ORT_PROVIDERS)
    vis = _sess(f"{MODEL_DIR}/vision-int4.onnx", ORT_PROVIDERS)
    pre = _sess(f"{MODEL_DIR}/baberu-prefill.onnx", ORT_PROVIDERS)
    stp = _sess(f"{MODEL_DIR}/baberu-step.onnx", ORT_PROVIDERS)
    print(f"all sessions: {(time.perf_counter()-t_all)*1000:.0f}ms", flush=True)
    with open(_load(f"{MODEL_DIR}/vocab.json"), encoding="utf-8") as f:
        charset = json.load(f)
    id2ch, content = {}, set()
    for i, ch in enumerate(charset):
        id2ch[i + 4] = ch
        # pass 1 like baberuParseVocab: single alnum, minus the long-dash set
        if len(ch) == 1 and ch not in "ーｰ〜~" and re.match(r"[A-Za-z0-9]", ch):
            content.add(i + 4)
    # pass 2 like the isContentChar extension (reference unicodedata approx)
    for i, ch in id2ch.items():
        cp = ord(ch[0]) if ch else 0
        if (re.match(r"[A-Za-z0-9]", ch) or 0x3040 <= cp <= 0x30FF
                or 0x3400 <= cp <= 0x9FFF or 0xF900 <= cp <= 0xFAFF
                or 0xFF66 <= cp <= 0xFF9D):
            content.add(i)
    baberu = {"vis": vis, "pre": pre, "stp": stp, "bos": 1, "eos": 2,
              "id2ch": id2ch, "contentIds": content}
    EPS.update({n: s.get_providers() for n, s in
                {"ctd": ctd, "vis": vis, "pre": pre, "stp": stp}.items()})
    print("ORT providers:", EPS, flush=True)


@app.on_event("startup")
def _startup():
    load_models()


@app.get("/health")
def health():
    return {"ok": bool(ctd and baberu), "device": os.environ.get("ORT_DEVICE", "cpu"),
            "panels": "client-fallback", "ep": EPS or None}


@app.get("/")
def root():
    return {"service": "arn-manga", "endpoints": ["/health", "POST /v1/page"]}


def nms(boxes, confs):
    idx = sorted(range(len(boxes)), key=lambda i: -confs[i])
    keep = []
    while idx:
        i = idx.pop(0)
        keep.append(i)
        a = boxes[i]
        rest = []
        for j in idx:
            b = boxes[j]
            ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
            iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
            inter = ix * iy
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if not (union > 0 and inter / union > NMS_THR):
                rest.append(j)
        idx = rest
    return keep


def split_tiles(w, h):
    vertical = h >= w
    long, short = (h, w) if vertical else (w, h)
    if long / short <= STRIP_ASPECT:
        return []
    step = TILE_SIZE - TILE_OVERLAP
    n = max(2, -(-(long - TILE_OVERLAP) // step))  # ceil
    ln = (long + (n - 1) * TILE_OVERLAP) / n
    out = []
    for i in range(n):
        o = round(i * (ln - TILE_OVERLAP))
        out.append((0, o, w, round(ln)) if vertical else (o, 0, round(ln), h))
    return out


def merge_tile_boxes(tiled):
    allb = [(b[0] + t[0], b[1] + t[1], b[2] + t[0], b[3] + t[1], b[4], ti)
            for ti, (t, bs) in enumerate(tiled) for b in bs]
    changed = True
    while changed:
        changed = False
        for i in range(len(allb)):
            for j in range(i + 1, len(allb)):
                a, b = allb[i], allb[j]
                if a[5] == b[5]:
                    continue
                gx = max(0, min(a[2], b[2]) - max(a[0], b[0]))
                gy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
                gapx = max(a[0] - b[2], b[0] - a[2], 0)
                gapy = max(a[1] - b[3], b[1] - a[3], 0)
                if not ((gx > 0 or gapx <= 8) and (gy > 0 or gapy <= 8)):
                    continue
                minside = min(a[2] - a[0], a[3] - a[1], b[2] - b[0], b[3] - b[1])
                if max(gx, gy) < 0.5 * minside:
                    continue
                allb[i] = (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]),
                           max(a[3], b[3]), max(a[4], b[4]), a[5])
                del allb[j]
                changed = True
                break
            if changed:
                break
    return [(x1, y1, x2, y2, c) for x1, y1, x2, y2, c, _ in allb]


def infer_once(pil, conf_thr):
    w, h = pil.size
    s = CTD_INPUT / max(w, h)
    nw, nh = round(w * s), round(h * s)
    canvas = Image.new("RGB", (CTD_INPUT, CTD_INPUT), LETTERBOX)
    canvas.paste(pil.resize((nw, nh), Image.BILINEAR), (0, 0))
    x = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    t0 = time.perf_counter()
    names = [o.name for o in ctd.get_outputs()]
    out = dict(zip(names, ctd.run(None, {"image": x})))
    infer_ms = (time.perf_counter() - t0) * 1000
    raw = out["bbox_preds"].reshape(-1, 7)
    boxes, confs, low_boxes, low_confs = [], [], [], []
    for cx, cy, bw, bh, c4, c5, c6 in raw:
        conf = float(c4 * max(c5, c6))
        if conf < 0.05:
            continue
        bx = [(cx - bw / 2) / s, (cy - bh / 2) / s,
              (cx + bw / 2) / s, (cy + bh / 2) / s]
        low_boxes.append(bx)
        low_confs.append(conf)
        if conf < conf_thr:
            continue
        boxes.append(bx)
        confs.append(conf)
    # mask: top-left nw×nh of the 1024 field, upscaled to page size
    m = out["mask"].reshape(CTD_INPUT, CTD_INPUT)[:nh, :nw]
    prob = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    return boxes, confs, low_boxes, low_confs, prob, infer_ms


def run_detect(pil, conf_thr, min_size):
    w, h = pil.size
    tiles = split_tiles(w, h)
    infer_ms = 0.0
    if not tiles:
        boxes, confs, low_boxes, low_confs, prob, infer_ms = infer_once(pil, conf_thr)
    else:
        per = []
        prob = np.zeros((h, w), np.float32)
        for (x0, y0, tw, th) in tiles:
            b, c, lb, lc, p, ms = infer_once(
                pil.crop((x0, y0, x0 + tw, y0 + th)), conf_thr)
            per.append(((x0, y0), b, c, lb, lc, p))
            infer_ms += ms
        merged = merge_tile_boxes(
            [((x0, y0), [(*bb[:4], cc) for bb, cc in zip(b, c)])
             for (x0, y0), b, c, _, _, _ in per])
        boxes = [[x1, y1, x2, y2] for x1, y1, x2, y2, _ in merged]
        confs = [c for _, _, _, _, c in merged]
        low_boxes, low_confs = [], []
        for (x0, y0), _, _, lb, lc, _ in per:
            for l, c in zip(lb, lc):
                low_boxes.append([l[0] + x0, l[1] + y0, l[2] + x0, l[3] + y0])
                low_confs.append(c)
        for (x0, y0), _, _, _, _, p in per:
            th, tw = p.shape
            np.maximum(prob[y0:y0 + th, x0:x0 + tw], p,
                       out=prob[y0:y0 + th, x0:x0 + tw])
    keep = [i for i in nms(boxes, confs)
            if boxes[i][2] - boxes[i][0] > min_size and boxes[i][3] - boxes[i][1] > min_size]
    # containment gate: ≥80% inside another → drop the lower-confidence one
    contained = set()
    for a in range(len(keep)):
        for b in range(len(keep)):
            if a == b or a in contained or b in contained:
                continue
            A, B = boxes[keep[a]], boxes[keep[b]]
            ix = max(0, min(A[2], B[2]) - max(A[0], B[0]))
            iy = max(0, min(A[3], B[3]) - max(A[1], B[1]))
            inter = ix * iy
            if not inter:
                continue
            aA = (A[2] - A[0]) * (A[3] - A[1])
            aB = (B[2] - B[0]) * (B[3] - B[1])
            if inter > 0.8 * aA:
                contained.add(a if confs[keep[a]] <= confs[keep[b]] else b)
            elif inter > 0.8 * aB:
                contained.add(b if confs[keep[b]] < confs[keep[a]] else a)
    out_boxes = [
        {"x1": max(0.0, boxes[i][0]), "y1": max(0.0, boxes[i][1]),
         "x2": min(float(w), boxes[i][2]), "y2": min(float(h), boxes[i][3]),
         "conf": confs[i]}
        for i in keep if i not in contained
    ]

    def overlaps(c):
        for o in out_boxes:
            ix = max(0, min(o["x2"], c[2]) - max(o["x1"], c[0]))
            iy = max(0, min(o["y2"], c[3]) - max(o["y1"], c[1]))
            inter = ix * iy
            if inter > 0.05 * (c[2] - c[0]) * (c[3] - c[1]) or \
               inter > 0.15 * (o["x2"] - o["x1"]) * (o["y2"] - o["y1"]):
                return True
        return False

    # mask components (4-connectivity like the browser BFS)
    packed = (prob > MASK_THR).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(packed, connectivity=4)
    prob_sum = np.bincount(labels.ravel(), weights=prob.ravel(), minlength=n)
    comps = []
    for lab in range(1, min(n, 401)):
        x, y, bw, bh, area = (int(stats[lab, i]) for i in range(5))
        if bw >= 8 and bh >= 8:
            comps.append([x, y, x + bw, y + bh, int(area), float(prob_sum[lab])])
    # merge touching-when-padded components
    changed = True
    while changed:
        changed = False
        for i in range(len(comps)):
            for j in range(i + 1, len(comps)):
                a, b = comps[i], comps[j]
                if not (a[0] - COMP_GAP > b[2] or b[0] - COMP_GAP > a[2]
                        or a[1] - COMP_GAP > b[3] or b[1] - COMP_GAP > a[3]):
                    comps[i] = [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]),
                                max(a[3], b[3]), a[4] + b[4], a[5] + b[5]]
                    del comps[j]
                    changed = True
                    break
            if changed:
                break
    page_area = w * h
    mask_boxes = []
    for x1, y1, x2, y2, count, psum in comps:
        if len(mask_boxes) >= 16:
            break
        bw, bh = x2 - x1, y2 - y1
        fill = count / (bw * bh)
        if bw < 14 or bh < 14 or fill < 0.02 or fill > 0.6:
            continue
        if bw * bh > 0.2 * page_area:
            continue
        if overlaps((x1, y1, x2, y2)):
            continue
        mask_prob = psum / count
        box_conf, c_area = 0.0, bw * bh
        for bx, bc in zip(low_boxes, low_confs):
            ix = max(0, min(bx[2], x2) - max(bx[0], x1))
            iy = max(0, min(bx[3], y2) - max(bx[1], y1))
            if ix * iy > 0.1 * c_area and bc > box_conf:
                box_conf = bc
        if mask_prob < 0.75 and box_conf < 0.20:
            continue
        mask_boxes.append({"x1": float(x1), "y1": float(y1),
                           "x2": float(x2), "y2": float(y2), "conf": 0.5})
    return out_boxes + mask_boxes, infer_ms


def run_baberu(crop):
    B = baberu
    x = np.asarray(crop.resize((224, 224), Image.BICUBIC),
                   dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    mean = np.array(BABERU_MEAN, np.float32).reshape(1, 3, 1, 1)
    std = np.array(BABERU_STD, np.float32).reshape(1, 3, 1, 1)
    t0 = time.perf_counter()
    (embeds,) = B["vis"].run(["vision_embeds"], {"pixel_values": x})
    if not all(np.isfinite(embeds.flat[:16])):
        raise RuntimeError("baberu vision produced non-finite embeds")
    out = B["pre"].run(None, {
        "vision_embeds": embeds,
        "input_ids": np.array([[B["bos"]]], dtype=np.int64)})
    names = [o.name for o in B["pre"].get_outputs()]

    def last_logits(vals):
        lg, shape = vals[names.index("logits")], B["pre"].get_outputs()[names.index("logits")].shape
        return lg.reshape(-1, shape[-1])[-1].astype(np.float64)

    logits = last_logits(out)
    present = [out[names.index(n)] for n in PRESENT_OUT]
    seq, toks = [B["bos"]], []
    pos = embeds.shape[1] + 1
    for _ in range(128):
        for tid in set(seq):
            s = logits[tid]
            logits[tid] = s * 1.2 if s < 0 else s / 1.2
        if toks and toks[-1] in B["contentIds"]:
            last, run = toks[-1], 0
            for t in reversed(toks):
                if t != last:
                    break
                run += 1
            if run >= 12:
                logits[last] = -np.inf
        nxt = int(np.argmax(logits[1:]) + 1)
        if nxt == B["eos"]:
            break
        toks.append(nxt)
        seq.append(nxt)
        if len(toks) >= 128:
            break
        feed = {"input_ids": np.array([[nxt]], dtype=np.int64),
                "position_ids": np.array([[pos]], dtype=np.int64)}
        for nm, p in zip(PAST_IN, present):
            feed[nm] = p
        out = B["stp"].run(None, feed)
        snames = [o.name for o in B["stp"].get_outputs()]
        lg = out[snames.index("logits")]
        v = B["stp"].get_outputs()[snames.index("logits")].shape[-1]
        logits = lg.reshape(-1, v)[-1].astype(np.float64)
        present = [out[snames.index(n)] for n in PRESENT_OUT]
        pos += 1
    ms = (time.perf_counter() - t0) * 1000
    return "".join(B["id2ch"].get(t, "") for t in toks), ms


def baberu_crop(pil, b):
    pad = max(4, (b["y2"] - b["y1"]) * 0.10)
    x = max(0, int(b["x1"] - pad))
    y = max(0, int(b["y1"] - pad))
    w = min(pil.width - x, int(np.ceil(b["x2"] - b["x1"] + 2 * pad)))
    h = min(pil.height - y, int(np.ceil(b["y2"] - b["y1"] + 2 * pad)))
    return pil.crop((x, y, x + w, y + h))


@app.post("/v1/page")
async def page(req: Request,
               conf_thr: float = Query(CONF_THR), min_size: int = Query(MIN_SIZE)):
    t0 = time.perf_counter()
    raw = await req.body()
    body_ms = (time.perf_counter() - t0) * 1000
    try:
        pil = Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception as e:
        return JSONResponse({"ok": False, "error": f"bad image: {e} (got {len(raw)} bytes head={raw[:8].hex()})"}, 400)
    async with lock:
        boxes, det_ms = run_detect(pil, conf_thr, min_size)
        texts, ocr_ms = [], 0.0
        for b in boxes:
            try:
                t, ms = run_baberu(baberu_crop(pil, b))
            except Exception:
                t, ms = "", 0.0
            texts.append(t)
            ocr_ms += ms
    total = (time.perf_counter() - t0) * 1000
    return {
        "ok": True,
        "w": pil.width, "h": pil.height,
        "boxes": [{"x1": round(float(b["x1"]), 1), "y1": round(float(b["y1"]), 1),
                   "x2": round(float(b["x2"]), 1), "y2": round(float(b["y2"]), 1),
                   "conf": round(float(b["conf"]), 4)} for b in boxes],
        "panels": [],
        "panelSkipped": "cloud-v1: panel runs client-side, banding fallback applies",
        "texts": texts,
        "ms": {"body": round(body_ms, 1), "detect": round(det_ms, 1),
               "ocr": round(ocr_ms, 1), "total": round(total, 1)},
    }


## 4. Deploy

First run takes a few minutes. The two values for the extension print
at the end. Run it again to update — the old API key stops working,
paste the new one.


In [ ]:
import pathlib, re, secrets, subprocess
assert pathlib.Path("modal_app.py").exists() and pathlib.Path("app.py").exists(), \
    "server files missing — re-run Step 3"
API_KEY = secrets.token_hex(32)
subprocess.run(["modal", "secret", "create", "--force", "arn-manga-key",
                f"ARN_API_KEY={API_KEY}"], check=True)
out = subprocess.run(["modal", "deploy", "modal_app.py"],
                     capture_output=True, text=True)
out = (out.stdout + "\n" + out.stderr).strip() + "\n"
print(out[-1500:])
m = re.search(r"https://[a-z0-9-]+--arn\-manga(?:-[a-z0-9-]+)?\.modal\.run", out)
if not m:
    raise SystemExit("✗ deploy output has no endpoint URL — read the log above, "
                     "fix, then re-run this cell")
ENDPOINT = m.group(0)
print("=" * 60)
print("ENDPOINT:", ENDPOINT)
print("API KEY:", API_KEY)
print("=" * 60)


## 5. Test

Sends one small image through your server to confirm it answers.


In [ ]:
import io, json, urllib.request
assert 'ENDPOINT' in dir() and 'API_KEY' in dir(), 'run Step 4 first'
from PIL import Image  # preinstalled on Colab
img = Image.new('L', (64, 64), 128)
buf = io.BytesIO()
img.save(buf, "JPEG")
req = urllib.request.Request(
    ENDPOINT + "/v1/page", data=buf.getvalue(),
    headers={"Authorization": "Bearer " + API_KEY, "Content-Type": "image/jpeg"})
# first request may cold-start the GPU container (~1-2 min) — be patient
with urllib.request.urlopen(req, timeout=300) as r:
    res = json.load(r)
assert res.get("ok") and "boxes" in res, res
print(f"✓ Endpoint is live: {res['w']}x{res['h']}, "
      f"{len(res['boxes'])} boxes, {res['ms']['total']}ms total")
print("=" * 60)
print("ENDPOINT:", ENDPOINT)
print("API KEY:", API_KEY)
print("=" * 60)
print("Paste both into the extension: Options → Model → Where detection runs → Cloud,")
print("then press Test cloud & prewarm.")
print("You can now close this tab — the endpoint stays up on Modal.")


## If something goes wrong

- **Step 2 fails:** a 🔑 Secret is missing, misnamed, swapped, or revoked.
  Fix it, then Runtime → Run all again.
- **Step 4 seems stuck:** normal the first time (a few minutes). Wait.
- **To update later:** Runtime → Run all again. The address stays the same;
  paste the new API key into the extension.
- **To delete everything:** run `!modal app stop arn-manga` in a new cell,
  then delete the `arn-manga-key` secret on the Modal dashboard.
